# Split Item 1 into Sentences

| Item | Description |
|---|---|
| Goal | Split the Item 1 text of every sampled filing into sentences with the same segmenter and settings that produced the hearing-side sentences, so firm sentences and legislator statements are the same kind of unit. |
| Input | `output/T3.4.1/filings.csv` (the sampled filings) and `sp500_10k_item1.parquet` (the Item 1 text per filing) under `/project2/mmiric_1435/acroh/llm_method/data`. |
| Output | `<scratch>/T3.6 - Firm Sentences/Processed/sentences.parquet`: one row per kept sentence with `accessionNo`, `gvkey`, `fyear`, `sent_idx`, `sentence`; and `output/T3.6.1/sentence_counts.csv`: one row per filing with its sentence counts. |
| Unit | A sentence of a filing's Item 1. |
| Sample | The 18,779 filings of the all-years sample; the three-year sample is a subset and needs no separate split. |
| Rules | Split the text on blank lines into blocks; flatten whitespace inside each block; segment each block with `pysbd` 0.3.4, English, `clean=False`, exactly as the hearing turns were segmented. Drop sentences shorter than 30 characters, which are headings, list fragments, and table residue. Number the kept sentences within each filing from 0 in order of appearance. |
| Variables | • `sent_idx`: position of the sentence among the filing's kept sentences, from 0.<br>• `n_sentences`: kept sentences in the filing; `n_dropped_short`: sentences removed by the length rule. |

See `Topic Relevance Rebuild Plan.md` in the TEMP folder for how this table is used downstream. The split runs in parallel across the allocation's CPUs because one core needs about 75 minutes for the sample's 1.06 billion characters.


## Step 1. Import packages

Read the saved tables, segment with `pysbd`, and run the split in parallel.


In [1]:
import os
import re
import time
from multiprocessing import Pool
from pathlib import Path

import pandas as pd
import pysbd
from IPython.display import display


## Step 2. Define file locations and rules

The large sentence table goes to scratch; the per-filing counts go beside the notebook outputs. The segmenter version is pinned because segmentation defines the unit.


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
SOURCE_DATA_FOLDER = Path("/project2/mmiric_1435/acroh/llm_method/data")
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.6.1"  # the identifier names this notebook's output folder
FILINGS_FILE = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output/T3.4.1/filings.csv"
ITEM1_FILE = SOURCE_DATA_FOLDER / "interim/repo_data/raw/03_sp500_10k_item1/sp500_10k_item1.parquet"
PROCESSED_FOLDER = SCRATCH_FOLDER / "T3.6 - Firm Sentences" / "Processed"
OUTPUT_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output" / NOTEBOOK_NAME

PYSBD_VERSION = "0.3.4"  # the release the hearing side was segmented with
MIN_SENTENCE_CHARS = 30  # shorter strings are headings, list fragments, and table residue
N_WORKERS = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count()))
assert pysbd.__version__ == PYSBD_VERSION, f"pysbd {pysbd.__version__} found, {PYSBD_VERSION} required"
PROCESSED_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Sentence table: {PROCESSED_FOLDER}")
print(f"Outputs: {OUTPUT_FOLDER}; workers: {N_WORKERS}")


Sentence table: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP/T3.6 - Firm Sentences/Processed
Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.6.1; workers: 8


## Step 3. Load the sampled filings and their Item 1 text

Read the filing list from T3.4.1, then only those filings' text. Every sampled filing must have its text.


In [3]:
filings = pd.read_csv(FILINGS_FILE, dtype={"gvkey": str, "cik": str})
assert filings["accessionNo"].is_unique
item1 = pd.read_parquet(ITEM1_FILE, columns=["accessionNo", "item1"])
item1 = item1.loc[item1["accessionNo"].isin(filings["accessionNo"])]
filings = filings.merge(item1, on="accessionNo", how="left", validate="one_to_one")
assert filings["item1"].notna().all()
filings["chars"] = filings["item1"].str.len()
print(f"Filings: {len(filings):,}; firms: {filings['gvkey'].nunique():,}; characters: {filings['chars'].sum() / 1e9:.2f} billion")
display(filings[["accessionNo", "gvkey", "conm", "fyear", "chars"]].head(3))


Filings: 18,779; firms: 1,058; characters: 0.95 billion


,accessionNo,gvkey,conm,fyear,chars
0,0000912057-99-011120,001013,ADC TELECOMMUNICATIONS INC,1999,52556
1,0000912057-01-000926,001013,ADC TELECOMMUNICATIONS INC,2000,54194
2,0000912057-02-001614,001013,ADC TELECOMMUNICATIONS INC,2001,51154


## Step 4. Define the split and show it on one filing

One function because the same logic runs on every filing in parallel. It splits on blank lines, flattens whitespace inside each block, segments each block, and keeps sentences of at least 30 characters. The first filing shows what comes out.


In [4]:
segmenter = pysbd.Segmenter(language="en", clean=False)


def split_item1(text):
    """Item 1 text to (kept sentences, number of short sentences dropped)."""
    kept = []
    n_dropped_short = 0
    for block in re.split(r"\n\s*\n|\r\n\s*\r\n", text or ""):
        flat = re.sub(r"\s+", " ", block).strip()
        if not flat:
            continue
        for sentence in segmenter.segment(flat):
            sentence = sentence.strip()
            if len(sentence) >= MIN_SENTENCE_CHARS:
                kept.append(sentence)
            elif sentence:
                n_dropped_short += 1
    return kept, n_dropped_short


example = filings.iloc[0]
example_sentences, example_dropped = split_item1(example["item1"])
print(f"{example['conm']} {example['fyear']}: {len(example_sentences):,} sentences kept, {example_dropped} short strings dropped")
for sentence in example_sentences[:3]:
    print(f"  - {sentence[:200]}")


ADC TELECOMMUNICATIONS INC 1999: 285 sentences kept, 24 short strings dropped
  - ADC Telecommunications, Inc. ("ADC") was incorporated in Minnesota in 1953 as Magnetic Controls Company and changed its name to ADC Telecommunications, Inc. in 1985.
  - ADC offers a broad range of network equipment, software and integration services for broadband, multiservice networks that deliver Internet/data, video and voice communications over telephone, cable t
  - ADC's broadband, multiservice network solutions enable local access, high-speed transmission and software management of communications services from service providers to consumers and businesses over 


## Step 5. Split every filing in parallel

Each worker process segments whole filings; results come back in the input order. The elapsed time is printed for the record.


In [5]:
started = time.time()
with Pool(N_WORKERS) as pool:
    results = pool.map(split_item1, filings["item1"].tolist(), chunksize=25)
elapsed_minutes = (time.time() - started) / 60
assert len(results) == len(filings)
filings["n_sentences"] = [len(sentences) for sentences, _ in results]
filings["n_dropped_short"] = [n_dropped for _, n_dropped in results]
print(f"Split {len(filings):,} filings in {elapsed_minutes:.1f} minutes with {N_WORKERS} workers")
print(f"Sentences kept: {filings['n_sentences'].sum():,}; short strings dropped: {filings['n_dropped_short'].sum():,}")
print(f"Sentences per filing: min {filings['n_sentences'].min()}, median {int(filings['n_sentences'].median())}, max {filings['n_sentences'].max()}")


Split 18,779 filings in 9.2 minutes with 8 workers
Sentences kept: 5,647,505; short strings dropped: 1,433,145
Sentences per filing: min 5, median 252, max 4266


## Step 6. Build the sentence table

One row per kept sentence, numbered within its filing in order of appearance. Every filing must keep at least one sentence.


In [6]:
assert filings["n_sentences"].gt(0).all()
sentence_rows = []
for filing, (sentences, _) in zip(filings[["accessionNo", "gvkey", "fyear"]].itertuples(index=False), results):
    for sent_idx, sentence in enumerate(sentences):
        sentence_rows.append((filing.accessionNo, filing.gvkey, filing.fyear, sent_idx, sentence))
sentences = pd.DataFrame(sentence_rows, columns=["accessionNo", "gvkey", "fyear", "sent_idx", "sentence"])
del sentence_rows, results
assert len(sentences) == filings["n_sentences"].sum()
assert not sentences.duplicated(["accessionNo", "sent_idx"]).any()
print(f"Sentence rows: {len(sentences):,} from {sentences['accessionNo'].nunique():,} filings")
display(sentences.head(3))


Sentence rows: 5,647,505 from 18,779 filings


,accessionNo,gvkey,fyear,sent_idx,sentence
0,0000912057-99-011120,001013,1999,0,"ADC Telecommunications, Inc. (""ADC"") was incor..."
1,0000912057-99-011120,001013,1999,1,"ADC offers a broad range of network equipment,..."
2,0000912057-99-011120,001013,1999,2,"ADC's broadband, multiservice network solution..."


## Step 7. Summarize coverage by fiscal year

Filings and sentences per fiscal year, for the record.


In [7]:
coverage = filings.groupby("fyear").agg(filings=("accessionNo", "size"), sentences=("n_sentences", "sum"), median_sentences=("n_sentences", "median"))
coverage["median_sentences"] = coverage["median_sentences"].astype(int)
display(coverage.T)


fyear,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
filings,703,692,713,726,730,738,732,728,737,725,...,688,678,674,670,654,655,648,649,643,625
sentences,184239,182793,198450,214398,226532,247426,218331,211991,216449,215670,...,202198,195806,194403,190167,203834,209795,209216,204849,196658,183345
median_sentences,212,215,220,238,247,266,243,243,242,250,...,245,242,240,238,270,286,286,277,270,253


## Step 8. Save the sentence table and the per-filing counts

The sentence table goes to scratch as Parquet; the counts go beside the notebook outputs so the record survives a scratch purge.


In [8]:
sentences_file = PROCESSED_FOLDER / "sentences.parquet"
counts_file = OUTPUT_FOLDER / "sentence_counts.csv"
sentences.to_parquet(sentences_file, index=False)
filings[["accessionNo", "gvkey", "fyear", "chars", "n_sentences", "n_dropped_short"]].to_csv(counts_file, index=False)
print(f"Saved: {sentences_file} ({sentences_file.stat().st_size / 1e6:.0f} MB)")
print(f"Saved: {counts_file}")


Saved: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP/T3.6 - Firm Sentences/Processed/sentences.parquet (413 MB)
Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.6.1/sentence_counts.csv
